# Building Dense Polymer Melts with MuPT

This notebook builds a dense all-atom polymer melt from repeat-unit SMILES, initializes coordinates with MuPT's AA-DPD builder, and writes a temporary `.mupt.sdf` file for later use.

Environment setup from the `mupt-examples` root:

```bash
mamba env create -f conda-envs/mupt-demo-env.yml
mamba activate mupt-demo-env
cd mupt
git switch joelaforet/issue-77-aa-dpd-builder  # won't have to do this once #60 and #83 are merged to main
python -m pip install -e ./mupt
```

The local `./mupt` checkout should be on PR #83 or another branch containing `mupt.builders.all_atom_dpd`.

Make sure the kernel used to run this notebook is the `mupt-demo-env` we just set up.

In [ ]:
from pathlib import Path

from demo_utils import (
    DPDRunConfig,
    PolymerRecipe,
    build_mupt_melt,
    build_repeat_unit_lexicon,
    ensure_mupt_pr83,
    find_examples_root,
    load_repeat_unit_libraries,
    plan_box_from_density,
    plan_chains_for_box,
    run_aa_dpd_initializer,
    show_available_chemistries,
    summarize_aa_dpd,
    summarize_build_plan,
    write_demo_mupt_sdf,
    write_mupt_visualization_pdb,
    write_manifest,
)

mupt_path = ensure_mupt_pr83()
EXAMPLES_ROOT = find_examples_root()
OUTPUT_DIR = EXAMPLES_ROOT / 'examples_system' / 'eric_melt_demo_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Using MuPT from: {mupt_path}')

## Browse Sample Repeat-Unit Chemistries 
### (you can make your own, these are just some that we ship as examples)
MuPT uses labelled repeat-unit SMILES. The `*` atoms mark polymerization linkers, and atom-map labels give MuPT a consistent head-to-tail direction.

In [ ]:
library = load_repeat_unit_libraries()
print('Available chemistries:')
for name, units in library.items():
    if units:
        print(f'  {name}: {len(units)} repeat-unit templates')

In [ ]:
show_available_chemistries(
    library,
    chemistries=[
        'polystyrene',
        'polymethyl-methacrylate',
        'polyethersulfone',
        'polythiophene',
    ],
)

## Choose Chemistry and Composition

This cell is the main place to edit chemistry. The default is a polystyrene melt; try replacing the recipe with PMMA, PES/PSU, or another entry from the library.

In [ ]:
library['polythiophene']

In [ ]:
recipe = PolymerRecipe(
    name='polythiophene',
    repeat_unit_smiles=library['polythiophene'],
    head_unit='head_thiophene',
    tail_unit='tail_thiophene',
    mid_unit_distribution={'mid_thiophene': 1.0},
    resname_map={
        'head_thiophene': 'PHT',
        'mid_thiophene': 'PMT',
        'tail_thiophene': 'PTT',
    },
)

lexicon = build_repeat_unit_lexicon(recipe)
print(f'Built {len(lexicon)} repeat-unit templates for {recipe.name}.')

### Optional: Mix Repeat Units Across Chemistries

A `PolymerRecipe` is just a choice of head unit, tail unit, and middle-unit probability distribution. Those units do not need to come from the same named library chemistry. If the repeat units have compatible head/tail connector annotations and the force field can parameterize the resulting molecule, MuPT can build random copolymers that mix very different fragments.

The example below defines a random copolymer that uses a polystyrene head, a polyethersulfone tail, and middle units sampled from styrene, thiophene, and bisphenol-S fragments.

In [ ]:
# mixed_repeat_units = {
#     **library['polystyrene'],
#     **library['polythiophene'],
#     **library['polyethersulfone'],
# }

# Or if you don't like shorthand:

# mixed_repeat_units = {
#  'head_styrene': '[H:1]-C(c1ccccc1)[CH2:2]-*',
#  'mid_R_styrene': '*-[C@H:1](c1ccccc1)[CH2:2]-*',
#  'mid_achiral_styrene': '*-[CH:1](c1ccccc1)[CH2:2]-*',
#  'mid_S_styrene': '*-[C@@H:1](c1ccccc1)[CH2:2]-*',
#  'tail_styrene': '*-[C:1](c1ccccc1)=C(-[H:2])',
#  'head_thiophene': '[H]-[C:1]1=C-C=[C:2](-S-1)-*',
#  'mid_thiophene': '*-[C:1]1=C-C=[C:2](-S-1)-*',
#  'tail_thiophene': '*-[C:1]1=C-C=[C:2](-S-1)-[H]',
#  'head_bisphenol_S': '[H]-[O:1]c1ccc(cc1)S(=O)(=O)c1cc[c:2](cc1)-*',
#  'mid_bisphenol_S': '*-[O:1]c1ccc(cc1)S(=O)(=O)c1cc[c:2](cc1)-*',
#  'tail_bisphenol_S': '*-[O:1]c1ccc(cc1)S(=O)(=O)c1ccc(cc1)[O:2]-[H]',
#  'head_bisphenol_A': '[H]-[O:1]c1ccc(cc1)C(-C)(-C)c1cc[c:2](cc1)-*',
#  'mid_bisphenol_A': '*-[O:1]c1ccc(cc1)C(-C)(-C)c1cc[c:2](cc1)-*',
#  'tail_bisphenol_A': '*-[O:1]c1ccc(cc1)C(-C)(-C)c1ccc(cc1)[O:2]-[H]'
# }

# mixed_recipe = PolymerRecipe(
#     name='ps_polythiophene_pes_random_copolymer',
#     repeat_unit_smiles=mixed_repeat_units,
#     head_unit='head_styrene',
#     tail_unit='tail_bisphenol_S',
#     mid_unit_distribution={
#         'mid_achiral_styrene': 0.40,
#         'mid_thiophene': 0.30,
#         'mid_bisphenol_S': 0.30,
#     },
#     resname_map={
#         'head_styrene': 'PSH',
#         'mid_achiral_styrene': 'PSM',
#         'mid_thiophene': 'PTH',
#         'mid_bisphenol_S': 'PES',
#         'tail_bisphenol_S': 'PST',
#     },
# )

# mixed_lexicon = build_repeat_unit_lexicon(mixed_recipe)
# print(f'Built {len(mixed_lexicon)} mixed repeat-unit templates for {mixed_recipe.name}.')

# recipe = mixed_recipe
# lexicon = mixed_lexicon

## Choose a Sizing Workflow

Use `density_to_box` when you know how many chains you want. Use `box_density_to_chain_count` when you know the target box size and density.

In [ ]:
SIZING_MODE = 'density_to_box'  # or 'box_density_to_chain_count'
TARGET_DENSITY_G_CM3 = 1.05
CHAIN_LENGTH_RANGE = (10, 20)
RANDOM_SEED = 7

if SIZING_MODE == 'density_to_box':
    plan = plan_box_from_density(
        recipe,
        lexicon,
        n_chains=5,
        chain_length_range=CHAIN_LENGTH_RANGE,
        density_g_cm3=TARGET_DENSITY_G_CM3,
        random_seed=RANDOM_SEED,
    )
elif SIZING_MODE == 'box_density_to_chain_count':
    plan = plan_chains_for_box(
        recipe,
        lexicon,
        box_lengths_nm=(2.5, 2.5, 2.5),
        density_g_cm3=TARGET_DENSITY_G_CM3,
        chain_length_range=CHAIN_LENGTH_RANGE,
        random_seed=RANDOM_SEED,
    )
else:
    raise ValueError(f'Unknown SIZING_MODE: {SIZING_MODE}')

summarize_build_plan(plan)

## Build the MuPT Hierarchy

The helper below creates a role-aware `UNIVERSE -> SEGMENT -> RESIDUE -> PARTICLE` hierarchy. AA-DPD will place and relax this hierarchy in the planned box.

In [ ]:
system = build_mupt_melt(recipe, lexicon, plan)
print(system.hierarchy_summary(to_depth=2))

## Initialize Dense Coordinates with AA-DPD

This is an initialization step, not a production simulation. Note, if the DPD doesn't converge, the structure should still be okay for energy minimization. Raise an issue if you find a chemistry that causes EM to explode. The next notebook minimizes and runs short MD with OpenMM before analysis.

In [ ]:
dpd_config = DPDRunConfig(
    device='CPU',
    n_steps_max=50_000,
    n_steps_per_interval=1_000,
    random_seed=RANDOM_SEED,
)

dpd_result = run_aa_dpd_initializer(system, recipe, plan, dpd_config)
summarize_aa_dpd(system, dpd_result)

## Export for Simulation

The temporary `.mupt.sdf` file stores one SDF record per polymer chain with MuPT atom metadata. The next notebook loads this file back into MuPT, then hands it to OpenFF/OpenMM.

In [ ]:
sdf_path = write_demo_mupt_sdf(
    system,
    OUTPUT_DIR / f'{recipe.name}_dense_melt',
    resname_map=recipe.resname_map,
)
visualization_pdb = write_mupt_visualization_pdb(
    system,
    recipe.resname_map,
    OUTPUT_DIR / f'{recipe.name}_aa_dpd_centered_whole.pdb',
)
manifest_path = write_manifest(
    OUTPUT_DIR / 'dense_melt_manifest.json',
    sdf_path=sdf_path,
    visualization_pdb=visualization_pdb,
    recipe_name=recipe.name,
    resname_map=recipe.resname_map,
    target_density_g_cm3=plan.target_density_g_cm3,
    actual_density_g_cm3=plan.actual_density_g_cm3,
    box_lengths_nm=plan.box_lengths_nm,
    chain_lengths=plan.chain_lengths,
)
print(f'Wrote {sdf_path}')
print(f'Wrote visualization structure: {visualization_pdb}')
print('The next cell opens this structure in an embedded NGLView widget.')
print(f'Wrote {manifest_path}')

## View the AA-DPD Structure

We can use NGLView to view the system directly in this notebook. The file shown below is a PDB written from the MuPT Representation after AA-DPD initialization.

In [ ]:
import nglview as nv

view = nv.show_file(str(visualization_pdb), ext='pdb')
view.clear_representations()
view.add_representation('spacefill', radius_scale=0.25)
view.add_representation('licorice', radius=0.08)
view.add_unitcell()
view.center()
view

## Try It Yourself

Good workshop edits:

- Change `SIZING_MODE` to `box_density_to_chain_count`.
- Change `TARGET_DENSITY_G_CM3`.
- Swap the recipe to PMMA or a PES/PSU copolymer by changing the middle-unit probabilities.
- Increase `n_chains` or the box size after confirming the default runs on your machine.